# Silver — Data Quality & DLQ

Config-driven quality checks on `bronze.orders`, results logged to metadata, and failed records routed to the dead letter queue.

In [ ]:
import sys

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
REPO_ROOT = notebook_path.split("/notebooks/")[0]
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from pyspark.sql import functions as F
from src.quality.engine import run_quality_checks, get_failed_records, RESULTS_TABLE
from src.quality.dlq import route_to_dlq, dlq_summary, DLQ_TABLE
from src.quality.rules import ORDERS_DQ_RULES
from src.ingestion.idempotent_loader import save_run_report_to_volume

orders = spark.table("globalmart.bronze.orders")
print("Bronze orders count:", orders.count())

In [ ]:
# Run 6 quality checks (5 required + referential integrity)
dq_result = run_quality_checks(
    spark, orders, ORDERS_DQ_RULES, table_name="globalmart.bronze.orders"
)
print("Overall:", dq_result.overall_status)
display(spark.table(RESULTS_TABLE).filter(F.col("run_id") == dq_result.run_id))

In [ ]:
# DLQ test — route only intentionally bad rows (isolated from bronze duplicate order_ids)
bad1 = (
    orders.limit(1)
    .withColumn("order_id", F.lit(None).cast("string"))
    .withColumn("customer_id", F.lit("bad-cust-1"))
)
bad2 = (
    orders.limit(1)
    .withColumn("order_id", F.lit("bad-order-2"))
    .withColumn("customer_id", F.lit(None).cast("string"))
)
bad = bad1.unionByName(bad2, allowMissingColumns=True)

failed = get_failed_records(spark, bad, ORDERS_DQ_RULES, severities=("critical",))
routed = route_to_dlq(
    spark, failed,
    source_table="globalmart.bronze.orders",
    failure_reason="critical_dq_check_failed_test",
)
print(f"Routed {routed} test records to DLQ")
display(dlq_summary(spark))

In [ ]:
import json

report = {
    "run_id": dq_result.run_id,
    "overall_status": dq_result.overall_status,
    "critical_passed": dq_result.critical_passed,
    "checks": dq_result.results,
    "dlq_table": DLQ_TABLE,
}
print(json.dumps(report, indent=2, default=str))
try:
    save_run_report_to_volume(report, dbutils, "/Volumes/globalmart/metadata/run_reports/dq_latest.json")
except Exception as exc:
    print("Volume save skipped:", exc)